In [10]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import itertools
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, precision_score,
                             accuracy_score, recall_score, f1_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE

from preprocessing import prepare_data


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# kNN

In [2]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
K_GRID = list(range(3, 52, 2))
WEIGHTS = ["uniform", "distance"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]



def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras.
    A validation fica com os dados mais recentes de train_val."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit
    (aplicado também a *others). Devolve arrays NumPy."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    """Limiar que maximiza o F1 (escolhido na validation)."""
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def run_knn(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)          # scaling só com o train

    best = {"auc": -np.inf}
    for w in WEIGHTS:
        for k in K_GRID:
            m = KNeighborsClassifier(n_neighbors=k, weights=w).fit(Atr, ytr)
            p_val = m.predict_proba(Ava)[:, 1]
            auc = roc_auc_score(yva, p_val)
            if verbose:
                print(f"  w={w:8s} k={k:2d}  val AUC={auc:.4f}")
            if auc > best["auc"]:                       # empate: fica o 1º (k menor)
                best = {"auc": auc, "k": k, "weights": w, "p_val": p_val}

    # limiar de decisão para recall/precision/F1: escolhido na validation
    thr = best_f1_threshold(yva, best["p_val"])

    # modelo final: refit em train+val ou só em train
    if refit:
        Afit, Ate = standardize(Xtv, Xte)         # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr
    final = KNeighborsClassifier(n_neighbors=best["k"], weights=best["weights"])
    final.fit(Afit, yfit)
    p_test = final.predict_proba(Ate)[:, list(final.classes_).index(1)]
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "pos_train": int(ytr.sum()), "pos_val": int(yva.sum()),
            "pos_test": int(yte.sum()),
            "k": best["k"], "weights": best["weights"],
            "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0)}


if __name__ == "__main__":
    rows = [run_knn(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 200)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "k", "weights", "val_AUC", "test_AUC",
            "test_PR_AUC", "thr", "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

split_test          strategy  k  weights  val_AUC  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows 13 distance    0.780     0.725        0.392 0.055   0.857      0.316 0.462
    recent     drop_features 19  uniform    0.783     0.820        0.510 0.158   0.857      0.450 0.590
    recent     global_median 47 distance    0.745     0.779        0.535 0.122   0.714      0.357 0.476
    recent     season_median 45 distance    0.744     0.794        0.545 0.107   0.810      0.378 0.515
    recent doy_window_median 11 distance    0.777     0.780        0.536 0.079   0.905      0.404 0.559
stratified         drop_rows 43 distance    0.810     0.803        0.408 0.116   0.889      0.320 0.471
stratified     drop_features 37 distance    0.890     0.795        0.432 0.219   0.444      0.400 0.421
stratified     global_median 33  uniform    0.838     0.803        0.414 0.152   0.852      0.319 0.465
stratified     season_median 27  uniform    0.855     0.785     

# Naive Bayes

In [4]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
VAR_SMOOTHING = [1e-9, 1e-6, 1e-3, 1e-2, 1e-1, 1.0]
ALPHAS = [0.1, 1.0]                                     # suavização de Laplace (Bernoulli)
VARIANTS = ["gaussian", "mixed"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]
BIN_PREFIXES = ("season_", "cluster_")


# ----------------------------------------------------------------------------
# Funções Auxiliares (iguais às do kNN)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


# ----------------------------------------------------------------------------
# Naive Bayes misto: Gaussian (contínuas) + Bernoulli (binárias)
# ----------------------------------------------------------------------------
class MixedNB:
    """P(c|x) ∝ P(c) · Π_contínuas N(x_j|μ_cj,σ²_cj) · Π_binárias Bern(x_j|p_cj).
    O prior aparece nos dois estimadores, por isso subtrai-se uma vez."""

    def __init__(self, cont_idx, bin_idx, var_smoothing=1e-9, alpha=1.0):
        self.cont_idx, self.bin_idx = cont_idx, bin_idx
        self.var_smoothing, self.alpha = var_smoothing, alpha

    def fit(self, A, y):
        self.g_ = GaussianNB(var_smoothing=self.var_smoothing).fit(A[:, self.cont_idx], y)
        self.b_ = BernoulliNB(alpha=self.alpha, binarize=0.5).fit(A[:, self.bin_idx], y)
        self.classes_ = self.g_.classes_
        return self

    def predict_proba(self, A):
        jll = (self.g_.predict_joint_log_proba(A[:, self.cont_idx])
               + self.b_.predict_joint_log_proba(A[:, self.bin_idx])
               - np.log(self.g_.class_prior_))
        jll -= jll.max(axis=1, keepdims=True)           # estabilidade numérica
        p = np.exp(jll)
        return p / p.sum(axis=1, keepdims=True)


def make_model(variant, columns, var_smoothing, alpha):
    if variant == "gaussian":
        return GaussianNB(var_smoothing=var_smoothing)
    cols = list(columns)
    bin_idx = [i for i, c in enumerate(cols) if c.startswith(BIN_PREFIXES)]
    cont_idx = [i for i in range(len(cols)) if i not in bin_idx]
    return MixedNB(cont_idx, bin_idx, var_smoothing, alpha)


def p1(model, A):
    return model.predict_proba(A)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_nb(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    cols = Xtv.columns
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)                    # scaling só com o train

    best = {"auc": -np.inf}
    for variant in VARIANTS:
        alphas = ALPHAS if variant == "mixed" else [None]
        for vs in VAR_SMOOTHING:
            for a in alphas:
                m = make_model(variant, cols, vs, a).fit(Atr, ytr)
                p_val = p1(m, Ava)
                auc = roc_auc_score(yva, p_val)
                if verbose:
                    print(f"  {variant:8s} vs={vs:.0e} a={a}  val AUC={auc:.4f}")
                if auc > best["auc"]:                   # empate: fica o 1º
                    best = {"auc": auc, "variant": variant, "var_smoothing": vs,
                            "alpha": a, "p_val": p_val}

    thr = best_f1_threshold(yva, best["p_val"])

    if refit:
        Afit, Ate = standardize(Xtv, Xte)               # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr
    final = make_model(best["variant"], cols, best["var_smoothing"], best["alpha"])
    final.fit(Afit, yfit)
    p_test = p1(final, Ate)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "variant": best["variant"], "var_smoothing": best["var_smoothing"],
            "alpha": best["alpha"], "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0)}


if __name__ == "__main__":
    rows = [run_nb(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 220)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "variant", "var_smoothing", "alpha",
            "val_AUC", "test_AUC", "test_PR_AUC", "thr", "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

split_test          strategy  variant  var_smoothing  alpha  val_AUC  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows    mixed          1.000    0.1    0.827     0.801        0.503 0.113   0.952      0.426 0.588
    recent     drop_features    mixed          1.000    1.0    0.812     0.799        0.442 0.083   1.000      0.412 0.583
    recent     global_median    mixed          1.000    1.0    0.810     0.797        0.487 0.090   1.000      0.412 0.583
    recent     season_median    mixed          1.000    1.0    0.807     0.799        0.489 0.094   1.000      0.420 0.592
    recent doy_window_median    mixed          1.000    1.0    0.806     0.799        0.489 0.071   1.000      0.396 0.568
stratified         drop_rows gaussian          0.000    NaN    0.876     0.752        0.341 1.000   0.815      0.289 0.427
stratified     drop_features gaussian          0.001    NaN    0.879     0.747        0.313 0.359   0.889      0.289 0.436
stratified     g

# SVM

In [7]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
C_GRID = [0.01, 0.1, 1, 10, 100]
GAMMA_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]              # só para o kernel rbf
CLASS_WEIGHTS = [None, "balanced"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]


def configs():
    """Grelha de hiperparâmetros (linear primeiro: em empates fica o mais simples)."""
    for cw in CLASS_WEIGHTS:
        for C in C_GRID:
            yield {"kernel": "linear", "C": C, "gamma": None, "class_weight": cw}
    for cw in CLASS_WEIGHTS:
        for C in C_GRID:
            for g in GAMMA_GRID:
                yield {"kernel": "rbf", "C": C, "gamma": g, "class_weight": cw}


def make_svc(cfg, probability=False):
    kw = {"kernel": cfg["kernel"], "C": cfg["C"],
          "class_weight": cfg["class_weight"],
          "probability": probability, "random_state": 0}
    if cfg["gamma"] is not None:
        kw["gamma"] = cfg["gamma"]
    return SVC(**kw)


# ----------------------------------------------------------------------------
# Auxiliares (iguais às do kNN / NB)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def p1(model, A):
    return model.predict_proba(A)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_svm(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)                    # scaling só com o train

    # 1) escolha dos hiperparâmetros pelo AUC (decision_function) na validation
    best = {"auc": -np.inf}
    for cfg in configs():
        m = make_svc(cfg).fit(Atr, ytr)
        auc = roc_auc_score(yva, m.decision_function(Ava))
        if verbose:
            print(f"  {cfg}  val AUC={auc:.4f}")
        if auc > best["auc"]:                           # empate: fica o 1º
            best = {"auc": auc, "cfg": cfg}
    cfg = best["cfg"]

    # 2) probabilidades de validation (Platt) -> limiar de F1
    m_val = make_svc(cfg, probability=True).fit(Atr, ytr)
    p_val = p1(m_val, Ava)
    thr = best_f1_threshold(yva, p_val)

    # 3) modelo final: refit em train+val ou só em train
    if refit:
        Afit, Ate = standardize(Xtv, Xte)               # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr
    final = make_svc(cfg, probability=True).fit(Afit, yfit)
    p_test = p1(final, Ate)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "kernel": cfg["kernel"], "C": cfg["C"], "gamma": cfg["gamma"],
            "class_weight": cfg["class_weight"],
            "val_AUC": best["auc"],
            "val_AUC_proba": roc_auc_score(yva, p_val),
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0)}


if __name__ == "__main__":
    rows = [run_svm(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "kernel", "C", "gamma", "class_weight",
            "val_AUC", "val_AUC_proba", "test_AUC", "test_PR_AUC", "thr",
            "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

split_test          strategy kernel   C  gamma class_weight  val_AUC  val_AUC_proba  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows    rbf 100  0.003     balanced    0.860          0.860     0.861        0.650 0.121   0.857      0.462 0.600
    recent     drop_features linear 100    NaN     balanced    0.864          0.864     0.879        0.660 0.090   1.000      0.438 0.609
    recent     global_median    rbf 100  0.010     balanced    0.854          0.853     0.827        0.639 0.074   0.905      0.422 0.576
    recent     season_median    rbf 100  0.010     balanced    0.847          0.847     0.827        0.614 0.213   0.619      0.542 0.578
    recent doy_window_median    rbf 100  0.003     balanced    0.861          0.861     0.854        0.658 0.170   0.762      0.485 0.593
stratified         drop_rows linear   1    NaN     balanced    0.890          0.890     0.823        0.398 0.162   0.778      0.339 0.472
stratified     drop_features    rb

# Random Forest

In [11]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
N_ESTIMATORS = 300
SEED = 0
MAX_DEPTH = [3, 5, 8, None]
MIN_LEAF = [1, 3, 5, 10]
MAX_FEATURES = ["sqrt", 0.5]
CLASS_WEIGHTS = [None, "balanced_subsample"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")


def configs():
    """Grelha (modelos mais simples primeiro: em empates fica o mais simples)."""
    for d, l, mf, cw in itertools.product(MAX_DEPTH, MIN_LEAF[::-1],
                                          MAX_FEATURES, CLASS_WEIGHTS):
        yield {"max_depth": d, "min_samples_leaf": l,
               "max_features": mf, "class_weight": cw}


def make_rf(cfg):
    return RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=SEED,
                                  n_jobs=-1, **cfg)


# ----------------------------------------------------------------------------
# Auxiliares (iguais às dos outros métodos)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def p1(model, X):
    return model.predict_proba(X)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_rf(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)

    best = {"auc": -np.inf}
    for cfg in configs():
        m = make_rf(cfg).fit(Xtr, ytr)
        p_val = p1(m, Xva)
        auc = roc_auc_score(yva, p_val)
        if verbose:
            print(f"  {cfg}  val AUC={auc:.4f}")
        if auc > best["auc"]:                           # empate: fica o 1º
            best = {"auc": auc, "cfg": cfg, "p_val": p_val}
    cfg = best["cfg"]

    thr = best_f1_threshold(yva, best["p_val"])

    Xfit, yfit = (Xtv, ytv) if refit else (Xtr, ytr)
    final = make_rf(cfg).fit(Xfit, yfit)
    p_test = p1(final, Xte)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "max_depth": cfg["max_depth"], "min_leaf": cfg["min_samples_leaf"],
            "max_features": cfg["max_features"], "class_weight": cfg["class_weight"],
            "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "accuracy": accuracy_score(yte, y_pred),
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0),
            "_importances": pd.Series(final.feature_importances_, index=Xtv.columns)}


if __name__ == "__main__":
    rows = [run_rf(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "max_depth", "min_leaf", "max_features",
            "class_weight", "val_AUC", "test_AUC", "test_PR_AUC", "thr", "accuracy",
            "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

    # importância média das variáveis (estratégias que usam CHL)
    for sp in SPLIT_TESTS:
        imp = pd.concat([r["_importances"] for r in rows
                         if r["split_test"] == sp and r["strategy"] != "drop_features"],
                        axis=1).mean(axis=1).sort_values(ascending=False)
        print(f"\nImportância média ({sp}):")
        print(imp.round(3).head(8).to_string())

split_test          strategy  max_depth  min_leaf max_features       class_weight  val_AUC  test_AUC  test_PR_AUC   thr  accuracy  recall  precision    f1
    recent         drop_rows        8.0         1         sqrt balanced_subsample    0.822     0.793        0.489 0.123     0.629   0.810      0.370 0.507
    recent     drop_features        8.0         3         sqrt balanced_subsample    0.812     0.848        0.555 0.279     0.663   0.905      0.404 0.559
    recent     global_median        8.0         1         sqrt balanced_subsample    0.805     0.804        0.516 0.159     0.629   0.857      0.375 0.522
    recent     season_median        8.0         1         sqrt balanced_subsample    0.798     0.817        0.561 0.109     0.596   1.000      0.368 0.538
    recent doy_window_median        5.0         3         sqrt balanced_subsample    0.802     0.816        0.514 0.322     0.663   0.905      0.404 0.559
stratified         drop_rows        3.0         1         sqrt        

# Logistic Regression

In [12]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
C_GRID = [0.001, 0.01, 0.1, 1, 10, 100]
PENALTIES = ["l2", "l1"]
CLASS_WEIGHTS = [None, "balanced"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent","recent20", "recent19", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]


def configs():
    """Grelha (C pequeno = mais regularização primeiro: em empates fica o mais simples)."""
    for pen in PENALTIES:
        for cw in CLASS_WEIGHTS:
            for C in C_GRID:
                yield {"penalty": pen, "C": C, "class_weight": cw}


def make_lr(cfg):
    solver = "lbfgs" if cfg["penalty"] == "l2" else "liblinear"
    return LogisticRegression(penalty=cfg["penalty"], C=cfg["C"],
                              class_weight=cfg["class_weight"],
                              solver=solver, max_iter=5000, random_state=0)


# ----------------------------------------------------------------------------
# Auxiliares (iguais às dos outros métodos)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def p1(model, A):
    return model.predict_proba(A)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_lr(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)                    # scaling só com o train

    # Apply SMOTE ONLY to the scaled training set
    smote = SMOTE(sampling_strategy="auto", k_neighbors=5, random_state=42)
    Atr_sm, ytr_sm = smote.fit_resample(Atr, ytr)

    print("Before SMOTE:", ytr.value_counts().to_dict())
    print("After SMOTE: ", pd.Series(ytr_sm).value_counts().to_dict())

    best = {"auc": -np.inf}
    for cfg in configs():
        m = make_lr(cfg).fit(Atr_sm, ytr_sm)
        p_val = p1(m, Ava)
        auc = roc_auc_score(yva, p_val)
        if verbose:
            print(f"  {cfg}  val AUC={auc:.4f}")
        if auc > best["auc"]:                           # empate: fica o 1º
            best = {"auc": auc, "cfg": cfg, "p_val": p_val}
    cfg = best["cfg"]

    thr = best_f1_threshold(yva, best["p_val"])

    if refit:
        Afit, Ate = standardize(Xtv, Xte)               # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr

    Afit_sm, yfit_sm = smote.fit_resample(Afit, yfit)
    final = make_lr(cfg).fit(Afit_sm, yfit_sm)
    p_test = p1(final, Ate)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "penalty": cfg["penalty"], "C": cfg["C"],
            "class_weight": cfg["class_weight"],
            "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "accuracy": accuracy_score(yte, y_pred),
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0),
            "_coef": pd.Series(final.coef_[0], index=Xtv.columns)}


if __name__ == "__main__":
    rows = [run_lr(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "penalty", "C", "class_weight",
            "val_AUC", "test_AUC", "test_PR_AUC", "thr", "accuracy",
            "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

    # coeficientes médios (estratégias que usam CHL), ordenados por |valor|
    for sp in SPLIT_TESTS:
        coef = pd.concat([r["_coef"] for r in rows
                          if r["split_test"] == sp and r["strategy"] != "drop_features"],
                         axis=1).mean(axis=1)
        print(f"\nCoeficientes médios ({sp}):")
        print(coef.reindex(coef.abs().sort_values(ascending=False).index)
                  .round(2).head(8).to_string())

Before SMOTE: {0: 344, 1: 45}
After SMOTE:  {0: 344, 1: 344}


c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

Before SMOTE: {0: 666, 1: 78}
After SMOTE:  {0: 666, 1: 666}


c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

Before SMOTE: {0: 666, 1: 78}
After SMOTE:  {0: 666, 1: 666}


c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

... (31 outputs left out)

c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

Before SMOTE: {0: 658, 1: 86}
After SMOTE:  {0: 658, 1: 658}


c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

split_test          strategy penalty      C class_weight  val_AUC  test_AUC  test_PR_AUC   thr  accuracy  recall  precision    f1
    recent         drop_rows      l2  10.00         None    0.856     0.857        0.606 0.201     0.640   1.000      0.396 0.568
    recent     drop_features      l1 100.00         None    0.876     0.872        0.645 0.418     0.730   1.000      0.467 0.636
    recent     global_median      l2 100.00         None    0.846     0.869        0.618 0.265     0.640   1.000      0.396 0.568
    recent     season_median      l2  10.00         None    0.842     0.868        0.618 0.272     0.652   1.000      0.404 0.575
    recent doy_window_median      l1   1.00         None    0.853     0.881        0.634 0.292     0.685   1.000      0.429 0.600
  recent20         drop_rows      l2 100.00         None    0.853     0.854        0.602 0.210     0.678   1.000      0.429 0.600
  recent20     drop_features      l1 100.00         None    0.878     0.869        0.654 0

# More Logistic Regression

In [14]:
from itertools import product
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from preprocessing import prepare_data

C_GRID = [0.01, 0.03, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
PENALTIES = ["l2", "l1"]
CLASS_WEIGHTS = [None, "balanced"]
STRATEGIES = (
    "drop_rows",
    "drop_features",
    "global_median",
    "season_median",
    "doy_window_median",
)
SPLIT_TESTS = ("recent", "recent20", "recent19", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]


def _scale(X_fit: pd.DataFrame, X_other: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Standardiza as variáveis contínuas usando apenas média e desvio de X_fit."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    A_fit = X_fit.astype(float).copy()
    A_other = X_other.astype(float).copy()
    A_fit[cont] = (A_fit[cont] - mu) / sd
    A_other[cont] = (A_other[cont] - mu) / sd
    return A_fit.to_numpy(), A_other.to_numpy()


def run_lr(strategy, split_test, n_val_years=3, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy,
        split_test,
        one_hot_cluster=True,
        sin_doy=True,
        season=season,
        scale=False,
        return_dates=True,
        **kw,
    )

    years = dtv.dt.year.to_numpy()
    y_arr = ytv.to_numpy()

    def _valid(mask):
        return mask.sum() >= 20 and np.unique(y_arr[mask]).size == 2

    # Expanding window: treino nos anos anteriores (years < yr), validação no ano yr
    valid_years = [
        yr for yr in np.unique(years)
        if _valid(years == yr) and _valid(years < yr)
    ]
    if valid_years:
        val_years = valid_years[-n_val_years:]
        folds = [(np.flatnonzero(years < yr), np.flatnonzero(years == yr)) for yr in val_years]
    else:
        # Fallback caso restem poucos anos (ex.: drop_rows + recent19): Leave-One-Year-Out
        valid_years = [
            yr for yr in np.unique(years)
            if _valid(years == yr) and _valid(years != yr)
        ]
        val_years = valid_years[-n_val_years:]
        folds = [(np.flatnonzero(years != yr), np.flatnonzero(years == yr)) for yr in val_years]

    scaled_folds = [
        (*_scale(Xtv.iloc[tr], Xtv.iloc[va]), y_arr[tr], y_arr[va])
        for tr, va in folds
    ]

    best = {"auc": -np.inf}
    for pen, cw, C in product(PENALTIES, CLASS_WEIGHTS, C_GRID):
        solver = "lbfgs" if pen == "l2" else "liblinear"
        fold_aucs, oof_y, oof_p = [], [], []

        for Atr, Ava, ytr, yva in scaled_folds:
            clf = LogisticRegression(
                penalty=pen, C=C, class_weight=cw, solver=solver, max_iter=5000, random_state=0
            ).fit(Atr, ytr)
            p_va = clf.predict_proba(Ava)[:, 1]
            fold_aucs.append(roc_auc_score(yva, p_va))
            oof_y.append(yva)
            oof_p.append(p_va)

        mean_auc = float(np.mean(fold_aucs))
        if verbose:
            print(f"  penalty={pen}, cw={cw}, C={C} -> val_AUC={mean_auc:.4f}")
        if mean_auc > best["auc"]:
            best = {
                "auc": mean_auc,
                "cfg": {"penalty": pen, "C": C, "class_weight": cw},
                "oof_y": np.concatenate(oof_y),
                "oof_p": np.concatenate(oof_p),
            }

    # Limiar que maximiza a Balanced Accuracy = (TPR + TNR) / 2 nas previsões out-of-fold
    fpr, tpr, thresholds = roc_curve(best["oof_y"], best["oof_p"])
    thr = float(thresholds[1:][np.argmax(tpr[1:] - fpr[1:])])

    # Ajuste final em todo o conjunto train_val e avaliação no test
    cfg = best["cfg"]
    solver = "lbfgs" if cfg["penalty"] == "l2" else "liblinear"
    Afit, Ate = _scale(Xtv, Xte)
    final = LogisticRegression(
        penalty=cfg["penalty"],
        C=cfg["C"],
        class_weight=cfg["class_weight"],
        solver=solver,
        max_iter=5000,
        random_state=0,
    ).fit(Afit, y_arr)

    p_test = final.predict_proba(Ate)[:, 1]
    y_pred = (p_test >= thr).astype(int)

    return {
        "strategy": strategy,
        "split_test": split_test,
        "val_years": ",".join(map(str, val_years)),
        "penalty": cfg["penalty"],
        "C": cfg["C"],
        "class_weight": cfg["class_weight"],
        "val_AUC": best["auc"],
        "test_AUC": roc_auc_score(yte, p_test),
        "test_PR_AUC": average_precision_score(yte, p_test),
        "thr": thr,
        "balanced_acc": balanced_accuracy_score(yte, y_pred),
        "recall": recall_score(yte, y_pred, zero_division=0),
        "precision": precision_score(yte, y_pred, zero_division=0),
        "f1": f1_score(yte, y_pred, zero_division=0),
        "_coef": pd.Series(final.coef_[0], index=Xtv.columns),
    }


if __name__ == "__main__":
    rows = [run_lr(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = [
        "split_test",
        "val_years",
        "strategy",
        "penalty",
        "C",
        "class_weight",
        "val_AUC",
        "test_AUC",
        "test_PR_AUC",
        "thr",
        "balanced_acc",
        "recall",
        "precision",
        "f1",
    ]
    print(res[show].round(3).to_string(index=False))

    # Coeficientes médios (estratégias que usam CHL), ordenados por |valor|
    for sp in SPLIT_TESTS:
        coef = pd.concat(
            [r["_coef"] for r in rows if r["split_test"] == sp and r["strategy"] != "drop_features"],
            axis=1,
        ).mean(axis=1)
        print(f"\nCoeficientes médios ({sp}):")
        print(coef.reindex(coef.abs().sort_values(ascending=False).index).round(2).head(8).to_string())

c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

split_test      val_years          strategy penalty    C class_weight  val_AUC  test_AUC  test_PR_AUC   thr  balanced_acc  recall  precision    f1
    recent 2016,2018,2019         drop_rows      l2 0.20     balanced    0.800     0.849        0.556 0.237         0.721   1.000      0.356 0.525
    recent 2016,2018,2019     drop_features      l1 0.10     balanced    0.858     0.854        0.618 0.552         0.798   0.905      0.475 0.623
    recent 2016,2018,2019     global_median      l1 0.20     balanced    0.837     0.856        0.599 0.431         0.798   0.905      0.475 0.623
    recent 2016,2018,2019     season_median      l1 0.20     balanced    0.835     0.858        0.595 0.439         0.791   0.905      0.463 0.613
    recent 2016,2018,2019 doy_window_median      l1 0.20     balanced    0.816     0.863        0.606 0.436         0.791   0.905      0.463 0.613
  recent20 2016,2018,2019         drop_rows      l2 0.20     balanced    0.800     0.846        0.558 0.237         0.

In [23]:
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Funções Auxiliares
# ----------------------------------------------------------------------------
def _doy365(date):
    """Dia do ano com 366 -> 365 (ano bissexto), como no cos de YDOY."""
    return np.minimum(date.dt.dayofyear.to_numpy(), 365)


# --- Imputação por regressão Ridge (implementada explicitamente em NumPy) ---
RIDGE_ALPHA_GRID = (1, 10, 30, 100, 300, 1000)
RIDGE_DEFAULT_ALPHA = 100.0   # usado se só houver 1 ano em train_val


def _impute_design(d, clusters, n_harmonics, use_env):
    """
    Matriz de features do imputador (usa só variáveis sempre observadas; NUNCA o HAB):
      - harmónicas sazonais sin/cos(2π·k·DOY/365), k = 1..n_harmonics
      - dummies do spatial_cluster (já com a fusão aplicada)
      - opcionalmente SST e UI
    """
    doy = _doy365(d["Date"])
    cols = []
    for k in range(1, n_harmonics + 1):
        ang = 2 * np.pi * k * doy / 365
        cols += [np.sin(ang), np.cos(ang)]
    sc = d["spatial_cluster"].to_numpy()
    cols += [(sc == c).astype(float) for c in clusters]
    if use_env:
        cols += [d["SST"].to_numpy(float), d["UI"].to_numpy(float)]
    return np.column_stack(cols)


def _ridge_fit(X, y, alpha):
    """Ridge em forma fechada: standardiza X, intercepto não penalizado."""
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd == 0] = 1.0
    Z = (X - mu) / sd
    y_mean = y.mean()
    # lstsq (mínima norma) em vez de solve: continua válido com alpha=0, em que as
    # dummies do cluster são colineares com o intercepto (matriz singular)
    w = np.linalg.lstsq(Z.T @ Z + alpha * np.eye(Z.shape[1]), Z.T @ (y - y_mean), rcond=None)[0]
    return {"mu": mu, "sd": sd, "w": w, "b": y_mean, "alpha": float(alpha)}


def _ridge_predict(model, X):
    return ((X - model["mu"]) / model["sd"]) @ model["w"] + model["b"]


def _select_alpha(X, y, years, grid):
    """Escolhe alpha por leave-one-year-out dentro de train_val (sem tocar no test)."""
    uy = np.unique(years)
    if len(uy) < 2:
        return RIDGE_DEFAULT_ALPHA
    sse = {a: 0.0 for a in grid}
    for yr in uy:
        te = years == yr
        for a in grid:
            m = _ridge_fit(X[~te], y[~te], a)
            sse[a] += np.sum((y[te] - _ridge_predict(m, X[te])) ** 2)
    return min(grid, key=lambda a: sse[a])


def _learn_fills(train_val, strategy, doy_half_window, clusters=(0, 1, 2, 4, 5),
                 impute_alpha="auto", impute_harmonics=2, impute_use_env=False):
    """Aprende, SÓ em train_val, as medianas / modelos usados na imputação."""
    fill_cols = ["CHL_log", "CHL_uncertainty"]
    obs = train_val.dropna(subset=fill_cols)
    fills = {"global": obs[fill_cols].median()}
    if strategy == "ridge_regression":
        X = _impute_design(obs, clusters, impute_harmonics, impute_use_env)
        years = obs["Date"].dt.year.to_numpy()
        fills["design"] = (tuple(clusters), impute_harmonics, impute_use_env)
        fills["ridge"], fills["bounds"] = {}, {}
        for c in fill_cols:                      # um modelo por variável a imputar
            y = obs[c].to_numpy(float)
            a = (_select_alpha(X, y, years, RIDGE_ALPHA_GRID)
                 if impute_alpha == "auto" else float(impute_alpha))
            fills["ridge"][c] = _ridge_fit(X, y, a)
            fills["bounds"][c] = (y.min(), y.max())   # previsões limitadas ao intervalo observado
    if strategy == "season_median":
        fills["season"] = obs.groupby("Season")[fill_cols].median()
    if strategy == "doy_window_median":
        fills["obs_doy"] = _doy365(obs["Date"])
        fills["obs_vals"] = obs[fill_cols].to_numpy()
        fills["half_window"] = doy_half_window
    return fills


def _apply_fills(d, strategy, fills):
    """Preenche CHL_log e CHL_uncertainty em falta (imputação em log(CHL))."""
    d = d.copy()
    fill_cols = ["CHL_log", "CHL_uncertainty"]
    miss = d[fill_cols].isna().any(axis=1).to_numpy()
    if not miss.any():
        return d

    if strategy == "global_median":
        for c in fill_cols:
            d[c] = d[c].fillna(fills["global"][c])

    elif strategy == "season_median":
        for c in fill_cols:
            by_season = d["Season"].map(fills["season"][c])
            d[c] = d[c].fillna(by_season).fillna(fills["global"][c])

    elif strategy == "ridge_regression":
        X = _impute_design(d, *fills["design"])
        for c in fill_cols:
            lo, hi = fills["bounds"][c]
            pred = np.clip(_ridge_predict(fills["ridge"][c], X), lo, hi)
            d[c] = d[c].fillna(pd.Series(pred, index=d.index))

    elif strategy == "doy_window_median":
        doy = _doy365(d["Date"])
        for i in np.flatnonzero(miss):
            diff = np.abs(fills["obs_doy"] - doy[i])
            diff = np.minimum(diff, 365 - diff)           # distância circular
            in_win = diff <= fills["half_window"]
            for j, c in enumerate(fill_cols):
                if np.isnan(d[c].iloc[i]):
                    v = (np.median(fills["obs_vals"][in_win, j])
                         if in_win.any() else fills["global"][c])
                    d.iloc[i, d.columns.get_loc(c)] = v
    return d

def _doy_window_median(
    obs_doy: np.ndarray, obs_vals: np.ndarray, target_doy: np.ndarray, half_window: int
) -> np.ndarray:
    """Calcula a mediana circular por DOY para cada linha em target_doy."""
    global_med = np.median(obs_vals, axis=0)
    out = np.empty((len(target_doy), obs_vals.shape[1]), dtype=float)
    for i, d in enumerate(target_doy):
        diff = np.abs(obs_doy - d)
        diff = np.minimum(diff, 365 - diff)
        in_win = diff <= half_window
        out[i] = np.median(obs_vals[in_win], axis=0) if in_win.any() else global_med
    return out













# ----------------------------------------------------------------------------
# Função principal
# ----------------------------------------------------------------------------
VALID_STRATEGIES = {
    "drop_rows",
    "drop_features",
    "global_median",
    "season_median",
    "doy_window_median",
    "ridge_regression",
}
VALID_SPLITS = {"recent", "recent19", "recent20", "stratified"}
VALID_SEASONS = {"one_hot", "drop"}



def prepare_data(strategy,
    split_test,
    one_hot_cluster=True,
    sin_doy=True,
    season="drop",
    scale=False,
    test_frac=0.15,
    doy_half_window=15,
    seed=42,
    return_dates=False,
    return_complete_mask=False,
    merge_clusters_2_3=True,
    add_anomalies=True,
    impute_alpha="auto",
    impute_harmonics=2,
    impute_use_env=False,):
    """
    strategy : estratégia para os missing values (CHL, CHL_uncertainty)
        "drop_rows"         – remove linhas de train_val com missing
        "drop_features"     – remove CHL_log e CHL_uncertainty
        "global_median"     – mediana global (CHL em log)
        "season_median"     – mediana por estação do ano
        "doy_window_median" – mediana numa janela de ±doy_half_window dias
        "ridge_regression"  – regressão Ridge (uma por variável: CHL_log e CHL_uncertainty),
                              treinada só nas linhas observadas de train_val, a partir de
                              harmónicas sazonais do DOY + cluster (+ SST/UI opcional).
                              Nunca usa o HAB. Ver parâmetros impute_*.

    split_test : forma de separar o conjunto de test
        "recent"     – test_frac mais recentes, sem linhas com missing
        "recent19"   – ano de 2019 completo como test; apenas anos < 2019 no treino
        "recent20"   – ano de 2020 completo como test; apenas anos < 2020 no treino
        "stratified" – ~test_frac do total, só datas completas, espalhadas no tempo
                       e por estações, preservando a mistura de classes

    one_hot_cluster : bool – spatial_cluster -> 5 colunas 0/1 (se merge_clusters_2_3=True)
                             ou 6 colunas 0/1 (se merge_clusters_2_3=False)
    sin_doy : bool – acrescenta sin(2π·DOY/365)
    season : str – tratamento de Season
        "one_hot" – 4 colunas 0/1 (season_Winter/Spring/Summer/Fall)
        "drop"    – remove Season (YDOY e sin_doy já capturam a sazonalidade)
    scale : bool – standardiza as variáveis contínuas (SST, UI, YDOY, sin_doy,
        CHL_log, CHL_uncertainty) com média e desvio (ddof=0) de train_val.
        Nota: se fores dividir train_val em train/val fora desta função, usa
        scale=False e aplica o scaling apenas dentro de cada fold de treino.
    test_frac : float – fração aproximada de dados para test (em "recent" e "stratified")
    doy_half_window : int – meia-janela em dias para "doy_window_median"
    seed : int – semente aleatória para "stratified"
    return_dates : bool – se True, devolve também as datas (Series) de train_val e test
    merge_clusters_2_3 : bool – funde o cluster espacial 3 no cluster 2
    impute_alpha : "auto" ou float – penalização do Ridge (só strategy="ridge_regression").
        "auto" escolhe, para cada variável, o melhor valor de {1,10,30,100,300,1000} por
        leave-one-year-out dentro de train_val.
    impute_harmonics : int – nº de harmónicas sazonais (sin/cos de k·2π·DOY/365) no imputador
    impute_use_env : bool – se True, o imputador usa também SST e UI (por defeito False:
        na validação não melhoravam a previsão de CHL)

    Devolve:
    (X_train_val, y_train_val, X_test, y_test)
    ou, com return_dates=True:
    (X_train_val, y_train_val, X_test, y_test, dates_train_val, dates_test)
    """

    # Validação explícita dos argumentos
    if strategy not in VALID_STRATEGIES:
        raise ValueError(f"strategy inválida: {strategy!r}. Opções: {sorted(VALID_STRATEGIES)}")
    if split_test not in VALID_SPLITS:
        raise ValueError(f"split_test inválido: {split_test!r}. Opções: {sorted(VALID_SPLITS)}")
    if season not in VALID_SEASONS:
        raise ValueError(f"season inválido: {season!r}. Opções: {sorted(VALID_SEASONS)}")
    if not (0.0 < test_frac < 1.0):
        raise ValueError(f"test_frac deve estar entre 0 e 1 (recebido: {test_frac})")
    if doy_half_window < 0:
        raise ValueError(f"doy_half_window deve ser >= 0 (recebido: {doy_half_window})")
    if not (isinstance(impute_harmonics, (int, np.integer)) and impute_harmonics >= 0):
        raise ValueError(f"impute_harmonics deve ser um inteiro >= 0 (recebido: {impute_harmonics!r})")
    if not (impute_alpha == "auto" or (isinstance(impute_alpha, (int, float)) and impute_alpha >= 0)):
        raise ValueError(f"impute_alpha deve ser 'auto' ou um número >= 0 (recebido: {impute_alpha!r})")


    # Read data
    df = pd.read_excel("classification_task1_train.xlsx")
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Date", "spatial_cluster"], kind="stable").reset_index(drop=True)

    # Fusão dos clusters
    clusters = (0, 1, 2, 3, 4, 5)
    if merge_clusters_2_3:
        df["spatial_cluster"] = df["spatial_cluster"].replace({3: 2})
        clusters = (0, 1, 2, 4, 5)

    # Separar os dados de test dos restantes dados
    complete = df[["CHL", "CHL_uncertainty"]].notna().all(axis=1).to_numpy()

    if split_test == "recent":
        n_test = int(round(test_frac * len(df)))
        cutoff = df["Date"].iloc[len(df) - n_test]
        is_test = (df["Date"] >= cutoff).to_numpy()
        test_mask = is_test & complete
        rest_mask = ~is_test

    elif split_test in {"recent19", "recent20"}:
        year = {"recent19": 2019, "recent20": 2020}[split_test]
        years = df["Date"].dt.year.to_numpy()
        is_test = years == year
        test_mask = is_test & complete
        rest_mask = years < year

    elif split_test == "stratified":
        rng = np.random.default_rng(seed)
        d = df.assign(_complete=complete)

        by_date = d.groupby("Date").agg(
            n_rows=("HAB", "size"),
            all_complete=("_complete", "all"),
            hab=("HAB", "max"),
            season=("Season", "first"),
        )
        elig = by_date[by_date["all_complete"]]

        n_target_dates = int(round(test_frac * len(df) / elig["n_rows"].mean()))
        n_target_dates = min(n_target_dates, len(elig))

        chosen = []
        for _, g in elig.groupby(["season", "hab"]):
            if len(g) <= 1:
                continue 
            k = int(round(n_target_dates * len(g) / len(elig)))
            k = min(max(k, 1), len(g) - 1)  # >=1 no test e >=1 no train_val
            pos = ((rng.uniform() + np.arange(k)) * len(g) / k).astype(int)
            chosen.extend(g.index[pos])

        test_mask = df["Date"].isin(chosen).to_numpy()
        rest_mask = ~test_mask

    train_val, test = df[rest_mask].copy(), df[test_mask].copy()
    

    # log(CHL), sin_doy, one-hot do cluster
    # (aplicadas igualmente a train_val e test, para as colunas coincidirem) 
    # pode-se fazer isto porque não aprende nada
    train_val["CHL_log"] = np.log(train_val["CHL"]) 
    test["CHL_log"] = np.log(test["CHL"])                    
    if sin_doy:
        train_val["sin_doy"] = np.sin(2 * np.pi * _doy365(train_val["Date"]) / 365)
        test["sin_doy"] = np.sin(2 * np.pi * _doy365(test["Date"]) / 365)
    if one_hot_cluster:
        for c in clusters:                       
            train_val[f"cluster_{c}"] = (train_val["spatial_cluster"] == c).astype(int)
            test[f"cluster_{c}"] = (test["spatial_cluster"] == c).astype(int)



    # Registar quais linhas de train_val tinham CHL observado antes de imputar
    if strategy == "drop_rows":
        train_val = train_val.dropna(subset=["CHL_log", "CHL_uncertainty"]).copy()
        complete_tv = np.ones(len(train_val), dtype=bool)
    else:
        complete_tv = train_val[["CHL_log", "CHL_uncertainty"]].notna().all(axis=1).to_numpy()
        if strategy != "drop_features":
            fills = _learn_fills(train_val, strategy, doy_half_window, clusters)
            train_val = _apply_fills(train_val, strategy, fills)

    # Estratégia para missing values (só para train_val, porque o test já é completo)
    if strategy == "drop_rows":
        train_val = train_val.dropna(subset=["CHL_log", "CHL_uncertainty"])
    elif strategy != "drop_features":
        fills = _learn_fills(train_val, strategy, doy_half_window, clusters=clusters,
                             impute_alpha=impute_alpha, impute_harmonics=impute_harmonics,
                             impute_use_env=impute_use_env)
        train_val = _apply_fills(train_val, strategy, fills)


    # Anomalias face à mediana esperada para o dia do ano (aprendidas só em train_val)
    if add_anomalies:
        tv_doy = _doy365(train_val["Date"])
        te_doy = _doy365(test["Date"])

        sst_med_tv = _doy_window_median(
            tv_doy, train_val[["SST"]].to_numpy(), tv_doy, doy_half_window
        )[:, 0]
        sst_med_te = _doy_window_median(
            tv_doy, train_val[["SST"]].to_numpy(), te_doy, doy_half_window
        )[:, 0]
        train_val["SST_anom"] = train_val["SST"].to_numpy() - sst_med_tv
        test["SST_anom"] = test["SST"].to_numpy() - sst_med_te

        if strategy != "drop_features":
            obs_chl = train_val.loc[complete_tv]
            obs_doy = _doy365(obs_chl["Date"])
            obs_vals = obs_chl[["CHL_log"]].to_numpy()
            chl_med_tv = _doy_window_median(obs_doy, obs_vals, tv_doy, doy_half_window)[:, 0]
            chl_med_te = _doy_window_median(obs_doy, obs_vals, te_doy, doy_half_window)[:, 0]
            train_val["CHL_anom"] = train_val["CHL_log"].to_numpy() - chl_med_tv
            test["CHL_anom"] = test["CHL_log"].to_numpy() - chl_med_te

    # Colunas finais (as restantes são redundantes e são descartadas)
    if season == "one_hot":
        for d in (train_val, test):
            for sn in ["Winter", "Spring", "Summer", "Fall"]:
                d[f"season_{sn}"] = (d["Season"] == sn).astype(int)
        cols = [f"season_{sn}" for sn in ["Winter", "Spring", "Summer", "Fall"]]
    else:
        cols = []
    cols += ["SST", "UI", "YDOY"]    
    if sin_doy:
        cols.append("sin_doy")
    if add_anomalies:
        cols.append("SST_anom")
    if strategy != "drop_features":
        cols += ["CHL_log", "CHL_uncertainty"]
    cols += ([f"cluster_{c}" for c in clusters]
             if one_hot_cluster else ["spatial_cluster"])


    X_train_val, X_test = train_val[cols].copy(), test[cols].copy()

    # Scaling das features contínuas: parâmetros aprendidos só em train_val
    if scale:
        cont = [c for c in ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"] if c in cols]
        mu = X_train_val[cont].mean()
        sd = X_train_val[cont].std(ddof=0).replace(0, 1.0)
        X_train_val[cont] = (X_train_val[cont] - mu) / sd
        X_test[cont] = (X_test[cont] - mu) / sd

    out = (X_train_val, train_val["HAB"], X_test, test["HAB"])

    if return_dates:
        out += (train_val["Date"], test["Date"])
    if return_complete_mask:
        out += (complete_tv,)
    return out

























from itertools import combinations, product
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


C_GRID = [0.01, 0.03, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
PENALTIES = ["l1", "l2"]
CLASS_WEIGHTS = [None, "balanced"]
IMPUTED_WEIGHTS = [0.3, 0.6, 1.0]

STRATEGIES = (
    "drop_rows",
    "drop_features",
    "global_median",
    "season_median",
    "doy_window_median",
    "ridge_regression",
)
SPLIT_TESTS = ("recent", "recent20", "recent19", "stratified")
CONT_COLS = [
    "SST",
    "UI",
    "YDOY",
    "sin_doy",
    "SST_anom",
    "CHL_log",
    "CHL_uncertainty",
    "CHL_anom",
]


def _scale_and_expand(
    X_fit: pd.DataFrame, X_other: pd.DataFrame, add_interactions: bool = True
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Standardiza as contínuas apenas com média/desvio de X_fit e gera termos
    quadráticos e interações físicas/espaciais já sobre as variáveis escaladas.
    """
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def _transform(df: pd.DataFrame) -> pd.DataFrame:
        out = df.astype(float).copy()
        out[cont] = (out[cont] - mu) / sd

        if add_interactions:
            phys = [c for c in ["SST", "UI", "CHL_log"] if c in out.columns]
            clusters = [c for c in out.columns if c.startswith("cluster_")]
            new_cols = {}

            # Termos quadráticos (ex.: janela ótima de SST)
            for c in phys:
                new_cols[f"{c}_sq"] = out[c] ** 2

            # Interações físicas (ex.: SST x UI, SST x CHL_log, UI x CHL_log)
            for c1, c2 in combinations(phys, 2):
                new_cols[f"{c1}_x_{c2}"] = out[c1] * out[c2]

            # Interações espaciais por cluster
            for cl in clusters:
                for c in phys:
                    new_cols[f"{cl}_x_{c}"] = out[cl] * out[c]

            if new_cols:
                out = pd.concat([out, pd.DataFrame(new_cols, index=out.index)], axis=1)
        return out

    df_fit = _transform(X_fit)
    df_other = _transform(X_other)
    return df_fit.to_numpy(), df_other.to_numpy(), list(df_fit.columns)


def run_lr(
    strategy,
    split_test,
    n_val_years=3,
    season="one_hot",
    add_interactions=True,
    add_anomalies=True,
    verbose=False,
    **kw,
):
    Xtv, ytv, Xte, yte, dtv, _, complete_tv = prepare_data(
        strategy,
        split_test,
        one_hot_cluster=True,
        sin_doy=True,
        season=season,
        scale=False,
        return_dates=True,
        return_complete_mask=True,
        add_anomalies=add_anomalies,
        **kw,
    )

    years = dtv.dt.year.to_numpy()
    y_arr = ytv.to_numpy()

    def _valid(mask):
        return mask.sum() >= 20 and np.unique(y_arr[mask]).size == 2

    # Expanding window: treino nos anos anteriores (years < yr), validação no ano yr
    valid_years = [
        yr for yr in np.unique(years) if _valid(years == yr) and _valid(years < yr)
    ]
    if valid_years:
        val_years = valid_years[-n_val_years:]
        folds = [(np.flatnonzero(years < yr), np.flatnonzero(years == yr)) for yr in val_years]
    else:
        valid_years = [
            yr for yr in np.unique(years) if _valid(years == yr) and _valid(years != yr)
        ]
        val_years = valid_years[-n_val_years:]
        folds = [(np.flatnonzero(years != yr), np.flatnonzero(years == yr)) for yr in val_years]

    scaled_folds = [
        (
            *_scale_and_expand(Xtv.iloc[tr], Xtv.iloc[va], add_interactions=add_interactions)[:2],
            y_arr[tr],
            y_arr[va],
            complete_tv[tr],
        )
        for tr, va in folds
    ]

    # Só faz sentido afinar peso de imputação se houver linhas imputadas no treino
    has_imputed = (strategy not in {"drop_rows", "drop_features"}) and (not complete_tv.all())
    imp_weights = IMPUTED_WEIGHTS if has_imputed else [1.0]

    best = {"auc": -np.inf}
    for pen, cw, C, imp_w in product(PENALTIES, CLASS_WEIGHTS, C_GRID, imp_weights):
        solver = "lbfgs" if pen == "l2" else "liblinear"
        fold_aucs, oof_y, oof_p = [], [], []

        for Atr, Ava, ytr, yva, comp_tr in scaled_folds:
            sw = np.where(comp_tr, 1.0, imp_w) if has_imputed else None
            clf = LogisticRegression(
                penalty=pen, C=C, class_weight=cw, solver=solver, max_iter=5000, random_state=0
            ).fit(Atr, ytr, sample_weight=sw)
            p_va = clf.predict_proba(Ava)[:, 1]
            fold_aucs.append(roc_auc_score(yva, p_va))
            oof_y.append(yva)
            oof_p.append(p_va)

        mean_auc = float(np.mean(fold_aucs))
        if verbose:
            print(f"  pen={pen}, cw={cw}, C={C}, imp_w={imp_w} -> val_AUC={mean_auc:.4f}")
        if mean_auc > best["auc"]:
            best = {
                "auc": mean_auc,
                "cfg": {"penalty": pen, "C": C, "class_weight": cw, "imp_w": imp_w},
                "oof_y": np.concatenate(oof_y),
                "oof_p": np.concatenate(oof_p),
            }

    # Limiar que maximiza a Balanced Accuracy nas previsões out-of-fold
    fpr, tpr, thresholds = roc_curve(best["oof_y"], best["oof_p"])
    thr = float(thresholds[1:][np.argmax(tpr[1:] - fpr[1:])])

    # Ajuste final em todo o conjunto train_val e avaliação no test
    cfg = best["cfg"]
    solver = "lbfgs" if cfg["penalty"] == "l2" else "liblinear"
    Afit, Ate, feat_names = _scale_and_expand(
        Xtv, Xte, add_interactions=add_interactions
    )
    sw_full = np.where(complete_tv, 1.0, cfg["imp_w"]) if has_imputed else None
    final = LogisticRegression(
        penalty=cfg["penalty"],
        C=cfg["C"],
        class_weight=cfg["class_weight"],
        solver=solver,
        max_iter=5000,
        random_state=0,
    ).fit(Afit, y_arr, sample_weight=sw_full)

    p_test = final.predict_proba(Ate)[:, 1]
    y_pred = (p_test >= thr).astype(int)

    return {
        "strategy": strategy,
        "split_test": split_test,
        "val_years": ",".join(map(str, val_years)),
        "penalty": cfg["penalty"],
        "C": cfg["C"],
        "class_weight": cfg["class_weight"] or "none",
        "imp_w": cfg["imp_w"],
        "val_AUC": best["auc"],
        "test_AUC": roc_auc_score(yte, p_test),
        "test_PR_AUC": average_precision_score(yte, p_test),
        "thr": thr,
        "balanced_acc": balanced_accuracy_score(yte, y_pred),
        "recall": recall_score(yte, y_pred, zero_division=0),
        "precision": precision_score(yte, y_pred, zero_division=0),
        "f1": f1_score(yte, y_pred, zero_division=0),
        "_coef": pd.Series(final.coef_[0], index=feat_names),
    }


if __name__ == "__main__":
    rows = [run_lr(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = [
        "split_test",
        "val_years",
        "strategy",
        "penalty",
        "C",
        "class_weight",
        "imp_w",
        "val_AUC",
        "test_AUC",
        "test_PR_AUC",
        "thr",
        "balanced_acc",
        "recall",
        "precision",
        "f1",
    ]
    print(res[show].round(3).to_string(index=False))

    # Coeficientes médios (estratégias que usam CHL), ordenados por |valor|
    for sp in SPLIT_TESTS:
        coef = pd.concat(
            [r["_coef"] for r in rows if r["split_test"] == sp and r["strategy"] != "drop_features"],
            axis=1,
        ).mean(axis=1)
        print(f"\nCoeficientes médios ({sp}):")
        print(coef.reindex(coef.abs().sort_values(ascending=False).index).round(2).head(10).to_string())

c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

split_test      val_years          strategy penalty    C class_weight  imp_w  val_AUC  test_AUC  test_PR_AUC   thr  balanced_acc  recall  precision    f1
    recent 2016,2018,2019         drop_rows      l1 0.50         none    1.0    0.772     0.833        0.561 0.065         0.741   0.952      0.385 0.548
    recent 2016,2018,2019     drop_features      l1 0.10     balanced    1.0    0.842     0.842        0.593 0.465         0.801   1.000      0.438 0.609
    recent 2016,2018,2019     global_median      l1 0.10     balanced    1.0    0.822     0.827        0.586 0.471         0.774   0.857      0.462 0.600
    recent 2016,2018,2019     season_median      l1 0.10     balanced    1.0    0.823     0.833        0.593 0.463         0.767   0.857      0.450 0.590
    recent 2016,2018,2019 doy_window_median      l1 0.10     balanced    1.0    0.801     0.845        0.587 0.455         0.798   0.905      0.475 0.623
    recent 2016,2018,2019  ridge_regression      l1 0.10     balanced    1.0

c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_